# **Assertions for `app/data.py`**

Every function in `app/data.py` feeds the Streamlit app. This notebook runs each one and compares the result with a number we already know from the 2017 and 2018 reports or from the data itself. If a check fails, the notebook stops and prints which check and why.

| Section | Function checked |
|---|---|
| 1 | `load_all` |
| 2 | `month_labels` |
| 3 | `pick` |
| 4 | `build_tables` |
| 5 | `monthly_table` |
| 6 | `month_series` |
| 7 | `long_series` |
| 8 | `closed_and_partial` |
| 9 | `load_metrics` and `load_model` |

Run the cells from top to bottom. The last cell prints the pass count.

## **Step 0: Import all Necessary Libraries & Setup**

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# This notebook lives in notebooks/. The app folder sits one level up.
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "app"))

import data       # app/data.py. It also adds src/ to the path.
import analysis   # src/analysis.py

passed = []


def check(name, ok, detail=""):
    """Stop with a clear message when a check fails. Otherwise record a pass."""
    assert ok, f"FAIL: {name}. {detail}"
    passed.append(name)
    print(f"PASS  {name}")

## **Step 1: Load Data**
The cleaned table holds the open hours only. Closed hours are not in the file.

In [2]:
df = data.load_all()

check("8,465 open rows", len(df) == 8465, f"got {len(df)}")
check("month_label column added", "month_label" in df.columns)
check("first date is 1 Dec 2017", df["date"].min() == pd.Timestamp("2017-12-01"))
check("last date is 30 Nov 2018", df["date"].max() == pd.Timestamp("2018-11-30"))
check("no missing rentals", df["rented_bike_count"].notna().all())
check("rentals never negative", (df["rented_bike_count"] >= 0).all())
check("hours run 0 to 23", set(df["hour"].unique()) == set(range(24)))
check("weekday runs 0 to 6", set(df["weekday"].unique()) == set(range(7)))

PASS  8,465 open rows
PASS  month_label column added
PASS  first date is 1 Dec 2017
PASS  last date is 30 Nov 2018
PASS  no missing rentals
PASS  rentals never negative
PASS  hours run 0 to 23
PASS  weekday runs 0 to 6


## **Step 2: Month Lables**
Twelve labels in calendar order. December 2017 comes first.

In [3]:
labels = data.month_labels(df)

expected_labels = ["Dec 2017", "Jan 2018", "Feb 2018", "Mar 2018", "Apr 2018", "May 2018",
                   "Jun 2018", "Jul 2018", "Aug 2018", "Sep 2018", "Oct 2018", "Nov 2018"]
check("12 labels in calendar order", labels == expected_labels, f"got {labels}")

PASS  12 labels in calendar order


## **Step 3: Pick Check**
`year=None` and `months=None` mean no filter. Filters combine, so an impossible pair returns no rows.

In [4]:
check("no filter keeps every row", len(data.pick(df)) == 8465)
check("2017 holds 744 rows (December only, 31 days x 24 hours)", len(data.pick(df, year=2017)) == 744)
check("2018 holds 7,721 rows", len(data.pick(df, year=2018)) == 7721)
check("2017 + 2018 rows add up to all rows", 744 + 7721 == len(df))
check("one month, Jun 2018, holds 720 rows (30 days x 24 hours)",
      len(data.pick(df, months=["Jun 2018"])) == 720)
check("two months together",
      len(data.pick(df, months=["Jun 2018", "Jul 2018"])) == 720 + 744)
check("empty list means no month filter", len(data.pick(df, months=[])) == 8465)
check("Dec 2017 inside year 2018 returns nothing",
      len(data.pick(df, year=2018, months=["Dec 2017"])) == 0)

PASS  no filter keeps every row
PASS  2017 holds 744 rows (December only, 31 days x 24 hours)
PASS  2018 holds 7,721 rows
PASS  2017 + 2018 rows add up to all rows
PASS  one month, Jun 2018, holds 720 rows (30 days x 24 hours)
PASS  two months together
PASS  empty list means no month filter
PASS  Dec 2017 inside year 2018 returns nothing


## **Step 4: Table Check**
It returns a daily table and a daily weather table for any slice.

In [5]:
daily, weather = data.build_tables(df)

check("353 open dates in the daily table", len(daily) == 353, f"got {len(daily)}")
check("daily rentals add up to 6,172,314 (Dec 2017 plus 2018)",
      daily["daily_rentals"].sum() == 6172314, f"got {daily['daily_rentals'].sum()}")
check("daily total equals the hourly total", daily["daily_rentals"].sum() == df["rented_bike_count"].sum())
check("weather table has one row per open date", len(weather) == len(daily))

PASS  353 open dates in the daily table
PASS  daily rentals add up to 6,172,314 (Dec 2017 plus 2018)
PASS  daily total equals the hourly total
PASS  weather table has one row per open date


## **Step 5: Monthly Table Check**
It loops one year at a time, so December 2017 is never compared with November 2018. The first month of each year has no change figure.

In [6]:
monthly = data.monthly_table(df).set_index("month_label")

check("12 months", len(monthly) == 12)
check("open dates add up to 353", monthly["open_dates"].sum() == 353)
check("monthly totals add up to the hourly total",
      monthly["total_rentals"].sum() == df["rented_bike_count"].sum())
check("Dec 2017 has no change figure", pd.isna(monthly.loc["Dec 2017", "change_vs_prev_pct"]))
check("Jan 2018 has no change figure (December 2017 is a different year)",
      pd.isna(monthly.loc["Jan 2018", "change_vs_prev_pct"]))

# Figures from the 2018 report
EXPECTED_JUNE_MEAN = 29896
EXPECTED_JANUARY_MEAN = 4839
EXPECTED_MARCH_RISE_PCT = 126.4

june = round(monthly.loc["Jun 2018", "mean_daily_rentals"])
january = round(monthly.loc["Jan 2018", "mean_daily_rentals"])
march = round(monthly.loc["Mar 2018", "change_vs_prev_pct"], 1)

check("June 2018 mean per day", june == EXPECTED_JUNE_MEAN, f"expected {EXPECTED_JUNE_MEAN}, got {june}")
check("January 2018 mean per day", january == EXPECTED_JANUARY_MEAN, f"expected {EXPECTED_JANUARY_MEAN}, got {january}")
check("March 2018 rise on February", march == EXPECTED_MARCH_RISE_PCT, f"expected {EXPECTED_MARCH_RISE_PCT}, got {march}")

# Figure from the 2017 report
check("December 2017 total is 185,330", monthly.loc["Dec 2017", "total_rentals"] == 185330)

highlights = analysis.monthly_highlights(monthly[monthly["year"] == 2018].reset_index())
check("highest month is June", highlights["highest_month"] == "June")
check("lowest month is January", highlights["lowest_month"] == "January")
check("biggest rise is March", highlights["biggest_monthly_rise"] == "March")
highlights

PASS  12 months
PASS  open dates add up to 353
PASS  monthly totals add up to the hourly total
PASS  Dec 2017 has no change figure
PASS  Jan 2018 has no change figure (December 2017 is a different year)
PASS  June 2018 mean per day
PASS  January 2018 mean per day
PASS  March 2018 rise on February
PASS  December 2017 total is 185,330
PASS  highest month is June
PASS  lowest month is January
PASS  biggest rise is March


{'biggest_monthly_rise': 'March',
 'biggest_rise_pct': 126.40817383615234,
 'lowest_month': 'January',
 'lowest_mean': 4838.903225806452,
 'highest_month': 'June',
 'highest_mean': 29896.233333333334}

## **Step 6: Month Series Check**
It gives every calendar date of one month. A closed date shows as a gap (NaN) and never as zero.

In [7]:
def series_for(label):
    daily_month, _ = data.build_tables(data.pick(df, months=[label]))
    return data.month_series(daily_month)

jun, sep, oct_, dec = (series_for(m) for m in ["Jun 2018", "Sep 2018", "Oct 2018", "Dec 2017"])

check("June has 30 dates and no gap", len(jun) == 30 and jun.isna().sum() == 0)
check("September has 30 dates and 4 gaps (26 open)", len(sep) == 30 and sep.isna().sum() == 4)
check("October has 31 dates and 3 gaps (28 open)", len(oct_) == 31 and oct_.isna().sum() == 3)
check("December 2017 has 31 dates and no gap", len(dec) == 31 and dec.isna().sum() == 0)
check("a gap is NaN, never zero", (sep.dropna() > 0).all())
check("first date of June is 1 Jun 2018", jun.index.min() == pd.Timestamp("2018-06-01"))
check("last date of June is 30 Jun 2018", jun.index.max() == pd.Timestamp("2018-06-30"))
check("busiest day of June is 13 Jun 2018", jun.idxmax() == pd.Timestamp("2018-06-13"))

PASS  June has 30 dates and no gap
PASS  September has 30 dates and 4 gaps (26 open)
PASS  October has 31 dates and 3 gaps (28 open)
PASS  December 2017 has 31 dates and no gap
PASS  a gap is NaN, never zero
PASS  first date of June is 1 Jun 2018
PASS  last date of June is 30 Jun 2018
PASS  busiest day of June is 13 Jun 2018


## **Step 7: Long Series Check**
It gives every calendar date from the first to the last, plus a 7-day average.

In [8]:
full, average = data.long_series(daily)

check("365 calendar dates", len(full) == 365, f"got {len(full)}")
check("12 closed dates are gaps", full.isna().sum() == 12, f"got {full.isna().sum()}")
check("average has the same length", len(average) == len(full))
check("series sums to the daily total", full.sum() == daily["daily_rentals"].sum())

PASS  365 calendar dates
PASS  12 closed dates are gaps
PASS  average has the same length
PASS  series sums to the daily total


## **Step 8: Closed and Partial Date Check**
It counts closed dates and partial days in the chosen slice. Closed dates are the gaps. 6 Oct 2018 is the one partial day.

In [9]:
closed, partial = data.closed_and_partial(daily, df)

check("12 closed dates over the year", closed == 12, f"got {closed}")
check("1 partial day over the year", partial == 1, f"got {partial}")

jun_daily, _ = data.build_tables(data.pick(df, months=["Jun 2018"]))
closed_jun, partial_jun = data.closed_and_partial(jun_daily, data.pick(df, months=["Jun 2018"]))
check("June has no closed date and no partial day", (closed_jun, partial_jun) == (0, 0))

PASS  12 closed dates over the year
PASS  1 partial day over the year
PASS  June has no closed date and no partial day


## **Step 9: Evaluation Metrics Check**
The metrics file holds the scores the Predict page shows. The model must load and predict.

In [10]:
metrics = data.load_metrics()

check("metrics hold the expected keys",
      {"model", "train_period", "test_period", "test_rows", "features", "mae", "rmse", "r2",
       "mean_error", "baseline_mae", "rush_hour_mean_error"} <= set(metrics))
check("model is gradient boosting", metrics["model"] == "Gradient boosting")
check("MAE 220.9", round(metrics["mae"], 1) == 220.9, f"got {metrics['mae']}")
check("RMSE 291.8", round(metrics["rmse"], 1) == 291.8, f"got {metrics['rmse']}")
check("R2 0.73", round(metrics["r2"], 2) == 0.73, f"got {metrics['r2']}")
check("mean error is negative (the model predicts low)", metrics["mean_error"] < 0)
check("test rows 1,313", metrics["test_rows"] == 1313)
check("rush hour errors at 08:00 and 18:00",
      round(metrics["rush_hour_mean_error"]["08:00"], 1) == -515.4
      and round(metrics["rush_hour_mean_error"]["18:00"], 1) == -479.1)
check("the model beats the best baseline",
      metrics["mae"] < min(metrics["baseline_mae"].values()))
check("9 features", len(metrics["features"]) == 9, f"got {metrics['features']}")

model = data.load_model()
frame = pd.DataFrame([{
    "hour": h, "month": 6, "weekday": 2, "holiday_flag": 0, "temperature": 20.0,
    "humidity": 50, "wind_speed": 1.5, "rainfall": 0.0, "snowfall": 0.0,
} for h in range(24)])[metrics["features"]]
predictions = model.predict(frame)

check("model returns 24 predictions", predictions.shape == (24,))
check("predictions are finite numbers", np.isfinite(predictions).all())
check("evening peak above 04:00 trough", predictions[18] > predictions[4])

PASS  metrics hold the expected keys
PASS  model is gradient boosting
PASS  MAE 220.9
PASS  RMSE 291.8
PASS  R2 0.73
PASS  mean error is negative (the model predicts low)
PASS  test rows 1,313
PASS  rush hour errors at 08:00 and 18:00
PASS  the model beats the best baseline
PASS  9 features
PASS  model returns 24 predictions
PASS  predictions are finite numbers
PASS  evening peak above 04:00 trough


## **Final Result**

In [11]:
print(f"All {len(passed)} checks passed.")

All 61 checks passed.
